# BME 590 - Exercise 1 - Liquid Handling
**Professor:** Emma Chory, Ph.D.

**Authors:** 
Rick Wierenga, Joe Laforet, Stefan Golas, Ben Perry

---

## Liquid Handling Basics
Now that you have gotten an introduction to setting up a deck in PyLabRobot, this second lesson is going to cover everything liquids! From setting their inintial levels to moving them around to managing tips and cross-contamination!


### Adding Liquids to Lab Objects
Usually when you start an automated protocol, you will have reservoirs of liquid somewhere on the deck. In fancier setups, this liquid can be moved to the deck from a pump system or a robotic arm carrying a reservoir from temperature control. For these exercises we will assume the liquid was placed manually by the user (such as manual preparation of reagents or usage of a kit, as is common in many experiments)

Therefore, to begin, we will need to **set the state** of liquids in a starting set of wells or a reservoir.

To start, let's go ahead and set up a basic HamiltonSTAR deck with:

- 1 reservoir
- 1 12 tube rack
- 2 96 well plates
- 1 384 well plate
- 1 1000 uL tip rack
- 1 300 uL tip rack
- 1 10 uL tip rack

We will start with a function that will set up this deck for us. Run the code below and you should get a deck that looks like this!

<div>
<img src="../figs/liquid_deck.png" width="850"/>
</div>

In [88]:
from pylabrobot.resources import Deck
from pylabrobot.liquid_handling.backends.backend import LiquidHandlerBackend
from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import LiquidHandlerChatterboxBackend
from pylabrobot.resources.hamilton import STARDeck
from pylabrobot.visualizer.visualizer import Visualizer

from pylabrobot.resources import (
    TIP_CAR_480_A00,
    PLT_CAR_L5AC_A00,
    corning_96_wellplate_360ul_flat,
    corning_384_wellplate_112ul_flat,
    nest_1_reservoir_195ml,
    corning_12_wellplate_6point9ml_flat,
    HTF,
    LTF,
    STF
)

async def visualize_deck(deck: Deck,
                         backend: LiquidHandlerBackend):
    try:
        lh = LiquidHandler(backend=backend, deck=deck)
        vis = Visualizer(resource = lh)
        await lh.setup()
        await vis.setup()
        return lh
    except Exception as e:
        print(f"Error! Got excpetion: {e}")

async def make_liquid_handling_deck():
    deck = STARDeck()
    tip_carrier = TIP_CAR_480_A00(name="awesome tip carrier 96x5")
    plate_carrier = PLT_CAR_L5AC_A00(name = "awesome plate carrier")
    tip_rack_names = [
        "1000_tips_0",
        "1000_tips_1"
        "300_tips_0",
        "300_tips_1",
        "10_tips_0"
    ]
    tip_rack_instantiators = [
        HTF,
        HTF,
        STF,
        STF,
        LTF
    ]
    plate_names = [
        "reservoir_0",
        "12_plate_0",
        "96_plate_0",
        "96_plate_1",
        "384_plate_0"
    ]
    plate_instantiators = [
        nest_1_reservoir_195ml,
        corning_12_wellplate_6point9ml_flat,
        corning_96_wellplate_360ul_flat,
        corning_96_wellplate_360ul_flat,
        corning_384_wellplate_112ul_flat
    ]
    for i, (name, tip_rack) in enumerate(zip(tip_rack_names, tip_rack_instantiators)):
        tip_carrier[i] = tip_rack(name = name)
    for i, (name, plate) in enumerate(zip(plate_names, plate_instantiators)):
        plate_carrier[i] = plate(name = name)

    deck.assign_child_resource(plate_carrier, rails = 5)
    deck.assign_child_resource(tip_carrier, rails = 11)
    return deck

deck = await make_liquid_handling_deck()
lh = await visualize_deck(deck, LiquidHandlerChatterboxBackend())

Setting up the liquid handler.
Resource deck was assigned to the liquid handler.
Resource trash was assigned to the liquid handler.
Resource trash_core96 was assigned to the liquid handler.
Resource waste_block was assigned to the liquid handler.
Resource awesome plate carrier was assigned to the liquid handler.
Resource awesome tip carrier 96x5 was assigned to the liquid handler.
Websocket server started at http://127.0.0.1:2135
File server started at http://127.0.0.1:1351 . Open this URL in your browser.


Let's print a summary of our deck so far.

In [89]:
print(lh.deck.summary())

Rail  Resource                         Type           Coordinates (mm)
(-6)  ├── trash_core96                 Trash          (-58.200, 106.000, 229.000)
      │
(5)   ├── awesome plate carrier        PlateCarrier   (190.000, 063.000, 100.000)
      │   ├── reservoir_0              Plate          (194.000, 071.500, 181.600)
      │   ├── 12_plate_0               Plate          (194.000, 167.500, 183.660)
      │   ├── 96_plate_0               Plate          (194.000, 263.500, 182.600)
      │   ├── 96_plate_1               Plate          (194.000, 359.500, 182.600)
      │   ├── 384_plate_0              Plate          (194.000, 455.500, 183.360)
      │
(11)  ├── awesome tip carrier 96x5     TipCarrier     (325.000, 063.000, 100.000)
      │   ├── 1000_tips_0              TipRack        (331.200, 073.000, 214.950)
      │   ├── 1000_tips_1300_tips_0    TipRack        (331.200, 169.000, 214.950)
      │   ├── 300_tips_1               TipRack        (331.200, 265.000, 214.950)
      │   ├

#### Setting Liquid Volumes
In order to manually set liquid volumes, there are a couple of methods we can use. Since we are going to be adding liquids to the plates/reservoirs, let's go ahead and get the 12-well and 96-well plate using `get_resource` with their names.

In [90]:
plate_96 = lh.deck.get_resource("96_plate_0")
plate_12 = lh.deck.get_resource("12_plate_0")

The easiest way we can fill a specific level of liquid in a specific well is to access it by index.

Remember that a plate has wells as it's children, and we can access these by identifier using the `get_well` method. This method expects an identifier, which can be either an index (0 - 95) representing the well index, or it can be the actual letter-integer coordinate of the well (e.g. A1, B2, etc.)

We show both indexing methods below

In [91]:
well_1 = plate_96.get_well(0)
well_2 = plate_96.get_well("A1")
well_3 = plate_96.get_item("A1")
well_4 = plate_96.get_item(0)

print(well_1 == well_2 == well_3 == well_4)

True


To fill this well with liquid, we need to access its `tracker` attribute and specifically use the `set_liquids` method. This method expects a **tuple**, where the first index is the name or class of liquid and the second index is the volume, in uL, for example:
- **(None, 180)** - Unknown liquid - 180 uL volume
- **("Red Dye", 2000)** - Custom liquid named "Red Dye" - 2000 uL volume
- **(Liquid.OCTANOL, 360)** - Official PLR Liquid class for Octanol - 360 uL volume.

Let's go ahead and demonstrate some of these below

In [95]:
from pylabrobot.resources import Liquid # import for liquid class

# get wells of interest
unknown_liquid = (None, 180)
red_dye = ("Red Dye", 2000)
octanol = (Liquid.OCTANOL, 360)

plate_96.get_well("A1").tracker.set_liquids([unknown_liquid])
plate_96.get_well("B1").tracker.set_liquids([octanol])
plate_12.get_well("A1").tracker.set_liquids([red_dye])

Great! Now go look at the visualizer. You should see something gn like this:

<div>
<img src="../figs/filled_wells.png" width="200"/>
</div>

The visualizer will actually shade the liquid volume by the amount in a well, relative to that well's maximum volume! So you can see, well B1 is filled to capacity (360 uL / 360 uL), so it has the deepest red color, while well A1 only has 180 uL, so it is filled to half capacity. Finally, well A1 of the 12-well plate is filled the least (percentage-wise) so it has the lightest color.

Currently, only red is supported as the color of solvent, so you'll haave to use your imagination here if mixing liquids. We can also use the `add_liquid` and `remove_liquid` functions to achieve similar functionality as the `set_liquids` function. These should **not** be used in lieu of pipetting operations, but rather are meant to be used to represent manual operations outside the liquid handler.

**Note-** The `add_liquid` function expects the liquid identifier as the first argument and the amount as the second argument, so just unpack the tuple. The `remove_liquid` doesn't require a liquid identifier, just the amount ot remove. The `remove_liquid` function will lso return a List of Tuples of liquids and volume removed if you call the function, which may be useful to store as a variable, for example, if you have to manually move an entire well contents somewhere else manually during a protocol.

In [96]:
plate_96.get_well("A1").tracker.add_liquid(*unknown_liquid)
plate_12.get_well("A1").tracker.remove_liquid(volume = 2000)


[('Red Dye', 2000)]

Great! You should now have something that looks like this

<div>
<img src="../figs/add_and_remove_liquid.png" width="200"/>
</div>

Note, since we doubled the liquid in well A1, it is now also full! We have also removed all the liquids in well A1 of the 12-well plate.

Certainly, you could manually assign liquids to each well, but this would get cumbersome with a lot of wells you are trying to fill!

There are several options for traversing over wells using iterators. Let's see some examples below


#### Setting Tip Locations
##### Tip Fill/Removal

#### Traversal of Labware Spots
##### Universal Traversal

#### Aspiration Operations

#### Dispense Operations

#### Tip Return & Discarding

#### Trackers
Trackers in PyLabRobot are objects that keep track of the state of the deck throughout a protocol. Three types of trackers currently exist:

- **tip trackers** - tracks the presence of tips in tip racks and on the pipetting channels
- **volume trackers** - tracks the volume in pipetting tips and wells/reservoirs
- **cross contamination** - tracks whether a given action would result in two liquids mixing unintentionally, thus introducing contamination in a sample or tip.

Why do we need trackers? 
#### Tracker Error Demonstration

#### Other Errors 